In [ ]:
# November 2026 correction: historical saved scores are NOT rerun or verified.
# Data splits, tokenizer sizes and Transformer pooling have changed; retrain
# models before comparing performance or using an old checkpoint.
import itertools
import random
import datetime
import os
import json
import keras

import pandas as pd
import numpy as np
import tensorflow as tf
import sentencepiece as spm
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, ConfusionMatrixDisplay

from keras.utils import plot_model

from tokenization_and_embedding import TokenAndPositionEmbedding
from transformer_block import TransformerBlock
from attentionPooling1D import AttentionPooling1D
from strip_mask import StripMask
from masked_attention_pooling import MaskedAttentionPooling
from token_mask import TokenMask

tf.config.optimizer.set_jit(False)

# Dataset https://www.kaggle.com/code/aadyasingh55/model-training-of-tweet-emotion-classification
from data_splits import split_emotion_data
import hashlib
from pathlib import Path


In [ ]:
log_dir = "logs/fit/"
os.makedirs(log_dir, exist_ok=True)

def set_all_seeds(seed: int):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)

def get_model_paths(model_tag):
    results_path = f"{log_dir}/{model_tag}_v2_results.csv"
    done_path = f"{log_dir}/{model_tag}_v2_done_runs.jsonl"
    return results_path, done_path

def load_done_runs(done_path):
    done_runs = set()
    if os.path.exists(done_path):
        with open(done_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    done_runs.add(row["run_name"])
    return done_runs

def save_run_result(row, run_name, results_path, done_path):
    df_row = pd.DataFrame([row])
    header = not os.path.exists(results_path)
    df_row.to_csv(results_path, mode="a", header=header, index=False)

    with open(done_path, "a", encoding="utf-8") as f:
        f.write(json.dumps({"run_name": run_name}) + "\n")

# Callback factory
# This cell creates all callbacks used during training, including
# early stopping, TensorBoard logging, best/latest checkpoints,
# and CSV history logging for each individual run.

def make_callbacks(run_name):
    run_dir = os.path.join(log_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    early_stop_callback = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=run_dir, histogram_freq=1, write_graph=True, update_freq="epoch")
    best_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(run_dir, "best.keras"), monitor="val_loss", save_best_only=True)
    latest_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(run_dir, "latest.keras"), save_best_only=False, save_freq="epoch")
    csv_callback = tf.keras.callbacks.CSVLogger(filename=os.path.join(run_dir, "history.csv"), append=True)

    return [early_stop_callback, tensorboard_callback, best_checkpoint_callback, latest_checkpoint_callback, csv_callback]

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs:", gpus)
print("Log dir:", log_dir)

In [ ]:
df = pd.read_parquet('data.parquet')

print(df.head())
print(df['label'])

emotion_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
df['emotion'] = df['label'].map(emotion_map)
# Preview the updated dataframe
print(df[['text', 'emotion']].head())

In [ ]:
### Constants

# Training Vars
EPOCHS = 7
PATIENCE = 1
BATCH_SIZE = 128
LEARNING_RATE = 3e-4

# Model vars
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256
DENSE_UNITS = 128
DROPOUT = 0.1
SHUFFLE_BUFFER = 10000

SEEDS = [42, 7]

# Tokeniser / padding
PAD_ID = 3
BPE_PREFIX = "m_bpe_v2"
UNIGRAM_PREFIX = "m_uni_v2"

# Params Grid
sweep = {
    "num_layers": [1, 2], 
    "num_heads": [4], 
    "ff_dim": [256, 512], 
    "dropout": [0.1, 0.2], 
    "lr": [3e-4, 1e-4], 
    "weight_decay": [1e-4, 3e-4], 
    "dense_units": [128],
}

# sweep = {
#     "num_layers": [2],
#     "num_heads": [4],
#     "ff_dim": [256],
#     "dropout": [0.1],
#     "lr": [3e-4],
#     "dense_units": [128]
# }

# RNN Vars
rnn_sweep = {
    "rnn_units": [96, 128], 
    "rnn_dropout": [0.2], 
    "label_smoothing": [0.3, 0.05], 
    "lr": [3e-4], 
    "weight_decay": [1e-4, 3e-4], 
    "embed_dim": [128, 256],
}

# rnn_sweep = {
#     "rnn_units": [128],
#     "rnn_dropout": [0.3],
#     "label_smoothing": [0.3],
#     "weight_decay": [1e-4],
#     "lr": [3e-4],          
#     "embed_dim": [128],    
# }

bigru_att_sweep = {
    "rnn_units": [128, 64], 
    "rnn_dropout": [0.2, 0.3], 
    "label_smoothing": [0.2, 0.3], 
    "lr": [3e-4, 1e-4], 
    "weight_decay": [3e-4, 1e-4, 5e-4], 
    "embed_dim": [128, 256],
}

# {'model_tag': 'att', 'num_layers': 2, 'num_heads': 4, 'ff_dim': 256, 'dropout': 0.1, 'lr': 0.0003, 'dense_units': 128, 'seed': 42, 'macro_f1': 0.8986465331629758, 'run_name': 'att_layers2_heads4_ff256_drop0.1_lr0.0003_dense128_seed42'}
# {'model_tag': 'bigru', 'rnn_units': 128, 'rnn_dropout': 0.2, 'label_smoothing': 0.3, 'weight_decay': 0.0001, 'lr': 0.0003, 'embed_dim': 128, 'seed': 42, 'macro_f1': 0.9172934078333205, 'run_name': 'bigru_units128_drop0.2_smooth0.3_wd0.0001_lr0.0003_embed128_seed42'}
# {'model_tag': 'bigru_att', 'rnn_units': 128, 'rnn_dropout': 0.2, 'label_smoothing': 0.2, 'weight_decay': 0.0001, 'lr': 0.0003, 'embed_dim': 128, 'seed': 42, 'macro_f1': 0.9158256394848157, 'run_name': 'bigru_att_units128_drop0.2_smooth0.2_wd0.0001_lr0.0003_embed128_seed42'} 

bilstm_sweep = {
    "rnn_units": [96, 128], 
    "rnn_dropout": [0.2, 0.3], 
    "label_smoothing": [0.2, 0.3], 
    "lr": [3e-4], 
    "weight_decay": [1e-4, 3e-4], 
    "embed_dim": [128],
}

VOCAB_SIZE = 16000
# NUMBER_OF_LAYERS = 2 # not use while testing params table
TRAIN_TEXT = 'train_text.txt'
LABEL_FILE = 'train_labels.txt'
SAMPLE_TEXT = "This is a sample sentence used for BPE bits. It can be up to 128 characters long."
TOKEN_LENGTH = 128

In [5]:
### BASIC PRE PROCESSING
print(df.shape)
print(df.columns)
print(f"Data Types: {df.dtypes}")
print(f"Empty values {df.isna().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))

(416809, 3)
Index(['text', 'label', 'emotion'], dtype='str')
Data Types: text         str
label      int64
emotion      str
dtype: object
Empty values text       0
label      0
emotion    0
dtype: int64
Duplicates: 686
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64
label
1    0.338
0    0.291
3    0.138
4    0.114
2    0.083
5    0.036
Name: proportion, dtype: float64


In [ ]:
# Remove duplicated/contradictory texts BEFORE any train/val/test partition.
(X_train, X_val, X_test, y_train, y_val, y_test), split_stats = split_emotion_data(df)
print("Cleaning and split statistics:", split_stats)
print("Train/validation/test rows:", len(X_train), len(X_val), len(X_test))
print(f"Train distribution (%):\n{y_train.value_counts(normalize=True).sort_index() * 100}")
print(f"Validation distribution (%):\n{y_val.value_counts(normalize=True).sort_index() * 100}")
print(f"Test distribution (%):\n{y_test.value_counts(normalize=True).sort_index() * 100}")


In [7]:
text_lengths = X_train.str.split().apply(len)
print(text_lengths.describe())

count    333447.000000
mean         19.200329
std          11.046795
min           1.000000
25%          11.000000
50%          17.000000
75%          25.000000
max         178.000000
Name: text, dtype: float64


In [8]:
X_train_clean = X_train.str.strip()
X_val_clean = X_val.str.strip()
X_test_clean = X_test.str.strip()

X_train_clean.to_csv(TRAIN_TEXT, index=False, header=False)
print(f"Wrote {len(X_train_clean)} to file")

Wrote 333447 to file


In [ ]:
### V2 tokenisers: generated strictly from the current training split.
corpus_digest = hashlib.sha256(Path(TRAIN_TEXT).read_bytes()).hexdigest()

def load_or_fit_train_tokenizer(prefix, kind):
    model_file = Path(f"{prefix}.model")
    digest_file = Path(f"{prefix}.train.sha256")
    if model_file.exists():
        if not digest_file.exists() or digest_file.read_text().strip() != corpus_digest:
            raise RuntimeError(
                f"{model_file} does not match this training corpus. "
                "Choose a new prefix and retrain; do not reuse old vocabularies."
            )
    else:
        spm.SentencePieceTrainer.train(
            input=TRAIN_TEXT, model_prefix=prefix, vocab_size=VOCAB_SIZE,
            model_type=kind, pad_id=PAD_ID, unk_id=0, bos_id=1, eos_id=2
        )
        digest_file.write_text(corpus_digest + "\n")
    sp = spm.SentencePieceProcessor()
    sp.load(str(model_file))
    assert sp.pad_id() == PAD_ID and sp.get_piece_size() == VOCAB_SIZE
    return sp

sp_bpe = load_or_fit_train_tokenizer(BPE_PREFIX, "bpe")
sp_uni = load_or_fit_train_tokenizer(UNIGRAM_PREFIX, "unigram")

print(f"Vocab size: {sp_bpe.get_piece_size()}")
print(f"BPE Pieces: {sp_bpe.encode(SAMPLE_TEXT, out_type=str)[:50]}")
print(f"Uni Pieces: {sp_uni.encode(SAMPLE_TEXT, out_type=str)[:50]}")

In [11]:
def encode_and_pad(sp, texts):
    all_ids = []
    pad_id = sp.pad_id()  # should be 3 (trained with pad_id=3)

    for text in texts:
        ids = sp.encode(text, out_type=int)
        ids = ids[:TOKEN_LENGTH]

        while len(ids) < TOKEN_LENGTH:
            ids.append(pad_id)

        all_ids.append(ids)

    return np.array(all_ids, dtype=np.int32)

def encode_for_transformer(sp, texts):
    input_ids = encode_and_pad(sp, texts)

    # 1 for real tokens, 0 for pad tokens
    pad_id = sp.pad_id()
    attention_masks = (input_ids != pad_id).astype(np.int32)

    return input_ids, attention_masks

def encode_for_rnn(sp, texts):
    return encode_and_pad(sp, texts)

print("SentencePiece pad_id (bpe, uni):", sp_bpe.pad_id(), sp_uni.pad_id())
assert sp_bpe.pad_id() == PAD_ID and sp_uni.pad_id() == PAD_ID, f"Unexpected pad_id (expected {PAD_ID}). Got bpe={sp_bpe.pad_id()}, uni={sp_uni.pad_id()}"

SentencePiece pad_id (bpe, uni): 3 3


In [12]:
### Encode with both tokenizers

# BPE encodings
Xb_train, att_b_mask_train = encode_for_transformer(sp_bpe, X_train_clean.tolist())
Xb_val, att_b_mask_val = encode_for_transformer(sp_bpe, X_val_clean.tolist())
Xb_test, att_b_mask_test = encode_for_transformer(sp_bpe, X_test_clean.tolist())

print(f"BPE train shape: {Xb_train.shape, att_b_mask_train.shape}")
print(f"BPE val shape: {Xb_val.shape, att_b_mask_val.shape}")
print(f"BPE test shape: {Xb_test.shape, att_b_mask_test.shape}")

# UNI encodings
Xu_train = encode_for_rnn(sp_uni, X_train_clean.tolist())
Xu_val = encode_for_rnn(sp_uni, X_val_clean.tolist())
Xu_test = encode_for_rnn(sp_uni, X_test_clean.tolist())

print("UNI train:", Xu_train.shape)
print("UNI val:", Xu_val.shape)
print("UNI test:", Xu_test.shape)

# UNI attention masks so UNI can also be used by the transformer
att_u_mask_train = (Xu_train != PAD_ID).astype("int32")
att_u_mask_val = (Xu_val != PAD_ID).astype("int32")
att_u_mask_test = (Xu_test != PAD_ID).astype("int32")


### Dataset builders
def make_rnn_dataset(X_ids, y_vals, seed=42, training=True):
    ds = tf.data.Dataset.from_tensor_slices((X_ids, y_vals))
    if training:
        ds = ds.shuffle(SHUFFLE_BUFFER, seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def make_dataset(X_ids, X_mask, y_vals, seed=42, training=True):
    ds = tf.data.Dataset.from_tensor_slices(((X_ids, X_mask), y_vals))
    if training:
        ds = ds.shuffle(SHUFFLE_BUFFER, seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


### Normal datasets
# UNI for RNN-family models
uni_rnn_train = make_rnn_dataset(Xu_train, y_train.values, seed=42, training=True)
uni_rnn_val = make_rnn_dataset(Xu_val, y_val.values, training=False)
uni_rnn_test = make_rnn_dataset(Xu_test, y_test.values, training=False)

# BPE for transformer
bpe_train_data = make_dataset(Xb_train, att_b_mask_train, y_train.values, seed=42, training=True)
bpe_val_data = make_dataset(Xb_val, att_b_mask_val, y_val.values, training=False)
bpe_test_data = make_dataset(Xb_test, att_b_mask_test, y_test.values, training=False)


### Opposite-tokenizer datasets
# BPE for RNN-family models
bpe_rnn_train = make_rnn_dataset(Xb_train, y_train.values, seed=42, training=True)
bpe_rnn_val = make_rnn_dataset(Xb_val, y_val.values, training=False)
bpe_rnn_test = make_rnn_dataset(Xb_test, y_test.values, training=False)

# UNI for transformer
uni_train_data = make_dataset(Xu_train, att_u_mask_train, y_train.values, seed=42, training=True)
uni_val_data = make_dataset(Xu_val, att_u_mask_val, y_val.values, training=False)
uni_test_data = make_dataset(Xu_test, att_u_mask_test, y_test.values, training=False)

print("Created normal and opposite-tokenizer datasets")
print("BPE RNN train:", Xb_train.shape)
print("UNI attention train:", Xu_train.shape)

BPE train shape: ((333447, 128), (333447, 128))
BPE val shape: ((41681, 128), (41681, 128))
BPE test shape: ((41681, 128), (41681, 128))
UNI train: (333447, 128)
UNI val: (41681, 128)
UNI test: (41681, 128)


W0000 00:00:1776799962.529707  766314 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1776799962.773159  766314 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9130 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:0c:00.0, compute capability: 12.0a


Created normal and opposite-tokenizer datasets
BPE RNN train: (333447, 128)
UNI attention train: (333447, 128)


In [13]:
### Balance classes
classes = np.sort(y_train.unique())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)

class_weights = {int(k): float(v) for k, v in zip(classes, weights)}
print(f"Class weights: {class_weights}")


Class weights: {0: 0.5732343809631868, 1: 0.4924503557725537, 2: 2.010436638570343, 3: 1.2119880490251669, 4: 1.455973277443018, 5: 4.639714476540324}


In [14]:
num_classes = y.nunique()
bpe_vocab_size = sp_bpe.get_piece_size()
uni_vocab_size = sp_uni.get_piece_size()

print(f"BPE Vocab size: {bpe_vocab_size}")
print(f"UNI Vocab size: {uni_vocab_size}")
print(f"Number of classes: {num_classes}")

BPE Vocab size: 16000
UNI Vocab size: 16000
Number of classes: 6


In [ ]:
### BiGRU + Attention
def build_bigru_attention_classifier(vocab_size, num_classes, max_len=128, embed_dim=128, rnn_units=128, dropout=0.2):
    """
    Build a bidirectional GRU model with masked attention pooling.

    This model:
    1. Converts token IDs into dense embeddings.
    2. Processes the sequence in both directions with a GRU.
    3. Uses attention pooling to focus on the most informative tokens.
    4. Applies dense classification layers to predict the emotion class.

    Args:
        vocab_size: Size of the tokenizer vocabulary.
        num_classes: Number of output classes.
        max_len: Fixed input sequence length.
        embed_dim: Size of each token embedding vector.
        rnn_units: Number of GRU units in each direction.
        dropout: Dropout rate used after pooling and dense layers.

    Returns:
        A compiled Keras model architecture.
    """
    # Input token IDs, shape: (batch_size, max_len)
    input_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")

    # Create a mask so padding tokens are ignored later.
    mask = TokenMask(pad_id=PAD_ID, name="token_mask")(input_ids)

    # Convert integer token IDs into dense embedding vectors.
    X = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=False, name="token_embedding")(input_ids)

    # Process the sequence in both forward and backward directions.
    # return_sequences=True keeps one output vector per token position, # which is needed for attention pooling.
    gru = tf.keras.layers.GRU(rnn_units, return_sequences=True)
    X = tf.keras.layers.Bidirectional(gru, name="bigru")(X, mask=mask)

    # Remove any built-in Keras mask metadata before custom pooling.
    X = StripMask(name="strip_mask")(X)

    # Learn which token positions matter most, while ignoring padding.
    X = MaskedAttentionPooling(pad_id=PAD_ID, name="attention_pool")(X, mask=mask)

    # Regularisation and dense feature learning before classification.
    X = tf.keras.layers.Dropout(dropout)(X)
    X = tf.keras.layers.Dense(DENSE_UNITS, activation="relu")(X)
    X = tf.keras.layers.Dropout(dropout)(X)

    # Final softmax layer for multi-class prediction.
    output = tf.keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(X)

    return tf.keras.Model(inputs=input_ids, outputs=output, name="bigru_attention")


## BiGRU
def build_bigru_classifier(vocab_size, num_classes, max_len=128, embed_dim=128, rnn_units=128, dropout=0.2):
    """
    Build a plain bidirectional GRU classifier.

    This model:
    1. Converts token IDs into embeddings.
    2. Processes the sequence in both directions with a GRU.
    3. Uses the final sequence representation for classification.

    Args:
        vocab_size: Size of the tokenizer vocabulary.
        num_classes: Number of output classes.
        max_len: Fixed input sequence length.
        embed_dim: Size of each token embedding vector.
        rnn_units: Number of GRU units in each direction.
        dropout: Dropout rate used before and after the dense layer.

    Returns:
        A Keras model architecture.
    """
    # Input token IDs.
    input_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")

    # Create a mask so padding tokens do not affect the GRU output.
    mask = keras.ops.not_equal(input_ids, PAD_ID)

    # Convert token IDs into dense embedding vectors.
    X = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=False, name="token_embedding")(input_ids)

    # Bidirectional GRU over the sequence.
    # return_sequences=False returns one summary vector for the full input.
    gru = tf.keras.layers.GRU(rnn_units, return_sequences=False)
    X = tf.keras.layers.Bidirectional(gru, name="bigru")(X, mask=mask)

    # Dense classification head.
    X = tf.keras.layers.Dropout(dropout)(X)
    X = tf.keras.layers.Dense(DENSE_UNITS, activation="relu")(X)
    X = tf.keras.layers.Dropout(dropout)(X)

    # Final class probabilities.
    output = tf.keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(X)

    return tf.keras.Model(inputs=input_ids, outputs=output, name="bigru")


## Attention / Transformer model
def build_transformer_classifier(vocab_size, num_classes, num_layers=2, max_len=128, num_heads=4, ff_dim=256, rate=0.1, dense_units=128):
    """
    Build a transformer-based classifier.

    This model:
    1. Combines token and positional embeddings.
    2. Passes the sequence through one or more transformer blocks.
    3. Pools only unpadded token representations with masked average pooling.
    4. Uses dense layers for final classification.

    Args:
        vocab_size: Size of the tokenizer vocabulary.
        num_classes: Number of output classes.
        num_layers: Number of transformer blocks.
        max_len: Fixed input sequence length.
        num_heads: Number of attention heads in each transformer block.
        ff_dim: Hidden size of the transformer feed-forward network.
        rate: Dropout rate used in transformer and dense layers.
        dense_units: Number of units in the dense classification layer.

    Returns:
        A Keras model architecture.
    """
    # Token IDs and explicit attention mask.
    input_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
    input_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")

    # Create token embeddings and add positional information.
    embedding_layer = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim=EMBED_DIM)

    X = embedding_layer(input_ids)

    # Apply stacked transformer blocks.
    for _ in range(num_layers):
        X = TransformerBlock(embed_dim=EMBED_DIM, num_heads=num_heads, ff_dim=ff_dim, rate=rate)(X, tensor_mask=input_mask)

    # Pool only real tokens; attention masking alone does not mask the final mean.
    X = tf.keras.layers.GlobalAveragePooling1D()(X, mask=keras.ops.cast(input_mask, 'bool'))

    # Dense classification head.
    X = tf.keras.layers.Dropout(rate)(X)
    X = tf.keras.layers.Dense(dense_units, activation="relu")(X)
    X = tf.keras.layers.Dropout(rate)(X)

    # Final class probabilities.
    output = tf.keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(X)

    return tf.keras.Model(inputs=[input_ids, input_mask], outputs=output, name="att")


## BiLSTM
def build_bilstm_classifier(vocab_size, num_classes, max_len=128, embed_dim=128, rnn_units=128, dropout=0.2):
    """
    Build a bidirectional LSTM classifier.

    This model:
    1. Converts token IDs into embeddings.
    2. Processes the sequence in both directions with an LSTM.
    3. Uses the final sequence representation for classification.

    Args:
        vocab_size: Size of the tokenizer vocabulary.
        num_classes: Number of output classes.
        max_len: Fixed input sequence length.
        embed_dim: Size of each token embedding vector.
        rnn_units: Number of LSTM units in each direction.
        dropout: Dropout rate used before and after the dense layer.

    Returns:
        A Keras model architecture.
    """
    # Input token IDs.
    input_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")

    # Create a padding mask so padded tokens are ignored.
    mask = keras.ops.not_equal(input_ids, PAD_ID)

    # Convert token IDs into embeddings.
    X = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=False, name="token_embedding")(input_ids)

    # Bidirectional LSTM over the sequence.
    # return_sequences=False returns one final summary vector.
    lstm = tf.keras.layers.LSTM(rnn_units, return_sequences=False)
    X = tf.keras.layers.Bidirectional(lstm, name="bilstm")(X, mask=mask)

    # Dense classification head.
    X = tf.keras.layers.Dropout(dropout)(X)
    X = tf.keras.layers.Dense(DENSE_UNITS, activation="relu")(X)
    X = tf.keras.layers.Dropout(dropout)(X)

    # Final class probabilities.
    output = tf.keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(X)

    return tf.keras.Model(inputs=input_ids, outputs=output, name="bilstm")

In [16]:
def sparse_ce_with_label_smoothing(num_classes: int, label_smoothing: float):
    # label_smoothing in [0, 1)
    c = float(num_classes)
    eps = float(label_smoothing)

    def loss_fn(y_true, y_pred):
        # y_true: (batch,) integer class ids
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)

        # one-hot then smooth
        y = tf.one_hot(y_true, depth=num_classes, dtype=tf.float32)
        y = (1.0 - eps) * y + (eps / c)

        # categorical CE expects y_true as distribution + y_pred as probs
        return tf.keras.losses.categorical_crossentropy(y, y_pred)

    return loss_fn

def evaluate_macro_f1(model, dataset):
    y_true = []
    y_pred = []

    for batch in dataset:
        X, y = batch

        preds = model.predict(X, verbose=0)
        preds = np.argmax(preds, axis=1)

        y_true.extend(y.numpy())
        y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return f1_score(y_true, y_pred, average="macro")

In [17]:
def train_one_rnn_config(model_builder, vocab_size, num_classes, train_ds, val_ds, class_weights, token_length, embed_dim, rnn_units, rnn_dropout, lr, weight_decay, label_smoothing, run_name):
    model = model_builder(
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_len=token_length,
        embed_dim=embed_dim,
        rnn_units=rnn_units,
        dropout=rnn_dropout
    )

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=lr,
            weight_decay=weight_decay
        ),
        loss=sparse_ce_with_label_smoothing(num_classes, label_smoothing),
        metrics=["accuracy"]
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=make_callbacks(run_name),
        verbose=0
    )

    macro_f1 = evaluate_macro_f1(model, val_ds)
    return model, history, macro_f1

def train_one_config(vocab_size, num_classes, train_data, val_data, class_weights, token_length, num_layers, num_heads, ff_dim, dropout, lr, weight_decay, dense_units, run_name):
    run_dir = os.path.join(log_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    model = build_transformer_classifier(
        vocab_size=vocab_size,
        num_classes=num_classes,
        max_len=token_length,
        num_layers=num_layers,
        num_heads=num_heads,
        ff_dim=ff_dim,
        rate=dropout,
        dense_units=dense_units
    )

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay), 
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"])
    
    early_stop_callback = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=run_dir, histogram_freq=1, write_graph=True, update_freq="epoch")
    best_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=f"{run_dir}/best.keras", monitor="val_loss", save_best_only=True)
    latest_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=f"{run_dir}/latest.keras", save_best_only=False, save_freq="epoch")
    csv_callback = tf.keras.callbacks.CSVLogger(filename=f"{run_dir}/history.csv", append=True)

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=[early_stop_callback, tensorboard_callback, latest_checkpoint_callback, best_checkpoint_callback, csv_callback],
        verbose=0
    )

    macro_f1 = evaluate_macro_f1(model, val_data)

    return model, history, macro_f1

In [ ]:
def run_transformer_sweep(model_tag, sweep, y, train_data, val_data, vocab_size, token_length=TOKEN_LENGTH):
    results = []

    results_path, done_path = get_model_paths(model_tag)
    done_runs = load_done_runs(done_path)

    keys = list(sweep.keys())
    num_classes = y.nunique()

    best_model = None
    best_history = None
    best_macro_f1 = -1.0
    best_config = None

    for values in itertools.product(*[sweep[k] for k in keys]):
        config = dict(zip(keys, values))

        for seed in SEEDS:
            set_all_seeds(seed)

            run_name = (
                f"{model_tag}_v2_"
                f"layers{config['num_layers']}"
                f"_heads{config['num_heads']}"
                f"_ff{config['ff_dim']}"
                f"_drop{config['dropout']}"
                f"_lr{config['lr']}"
                f"_wd{config['weight_decay']}"
                f"_dense{config['dense_units']}"
                f"_seed{seed}"
            )

            print(f"Starting run: {run_name}")

            if run_name in done_runs:
                print(f"Skipping run {run_name}, already completed")
                continue

            model, history, macro_f1 = train_one_config(
                vocab_size=vocab_size,         
                num_classes=num_classes,         
                train_data=train_data,         
                val_data=val_data,         
                class_weights=class_weights,         
                token_length=token_length,         
                num_layers=int(config["num_layers"]),         
                num_heads=int(config["num_heads"]),         
                ff_dim=int(config["ff_dim"]),         
                dropout=float(config["dropout"]),         
                lr=float(config["lr"]),         
                weight_decay=float(config["weight_decay"]),         
                dense_units=int(config["dense_units"]),         
                run_name=run_name
            )

            row = {
                "model_tag": model_tag,         
                **config,         
                "seed": seed,         
                "macro_f1": float(macro_f1),         
                "run_name": run_name
            }

            results.append(row)
            save_run_result(row, run_name, results_path, done_path)
            done_runs.add(run_name)

            if macro_f1 > best_macro_f1:
                best_macro_f1 = macro_f1
                best_model = model
                best_history = history
                best_config = row

            print(f"Finished run: {run_name} | macro_f1={macro_f1:.4f}")

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values("macro_f1", ascending=False)

    return best_model, best_history, best_config, results_df

In [ ]:
def run_rnn_sweep(model_tag, model_builder, rnn_sweep, train_ds, val_ds, y, vocab_size, token_length=TOKEN_LENGTH):
    results = []

    results_path, done_path = get_model_paths(model_tag)
    done_runs = load_done_runs(done_path)

    rnn_keys = list(rnn_sweep.keys())
    num_classes = y.nunique()

    best_model = None
    best_history = None
    best_macro_f1 = -1.0
    best_config = None

    for values in itertools.product(*[rnn_sweep[k] for k in rnn_keys]):
        config = dict(zip(rnn_keys, values))

        for seed in SEEDS:
            set_all_seeds(seed)

            run_name = (
                f"{model_tag}_v2_"
                f"units{config['rnn_units']}"
                f"_drop{config['rnn_dropout']}"
                f"_smooth{config['label_smoothing']}"
                f"_wd{config['weight_decay']}"
                f"_lr{config['lr']}"
                f"_embed{config['embed_dim']}"
                f"_seed{seed}"
            )

            print(f"Starting run: {run_name}")

            if run_name in done_runs:
                print(f"Skipping run {run_name}, already completed")
                continue

            model, history, macro_f1 = train_one_rnn_config(
                model_builder=model_builder,
                vocab_size=vocab_size,
                num_classes=num_classes,
                train_ds=train_ds,
                val_ds=val_ds,
                class_weights=class_weights,
                token_length=token_length,
                embed_dim=int(config["embed_dim"]),
                rnn_units=int(config["rnn_units"]),
                rnn_dropout=float(config["rnn_dropout"]),
                lr=float(config["lr"]),
                weight_decay=float(config["weight_decay"]),
                label_smoothing=float(config["label_smoothing"]),
                run_name=run_name
            )

            row = {
                "model_tag": model_tag,
                **config,
                "seed": seed,
                "macro_f1": float(macro_f1),
                "run_name": run_name
            }

            results.append(row)
            save_run_result(row, run_name, results_path, done_path)
            done_runs.add(run_name)

            if macro_f1 > best_macro_f1:
                best_macro_f1 = macro_f1
                best_model = model
                best_history = history
                best_config = row

            print(f"Finished run: {run_name} | macro_f1={macro_f1:.4f}")

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values("macro_f1", ascending=False)

    return best_model, best_history, best_config, results_df

In [ ]:
# normal runs

# Transformer with BPE
att_model, history_att, best_att_config, att_results_df = run_transformer_sweep(
    model_tag="att_bpe",
    sweep=sweep, 
    y=y,
    train_data=bpe_train_data,
    val_data=bpe_val_data,
    vocab_size=bpe_vocab_size
)

# Plain BiGRU with UNI
bigru_model, history_bigru, best_bigru_config, bigru_results_df = run_rnn_sweep(
    model_tag="bigru_uni",
    model_builder=build_bigru_classifier,
    rnn_sweep=rnn_sweep,
    train_ds=uni_rnn_train,
    val_ds=uni_rnn_val,
    y=y,
    vocab_size=uni_vocab_size
)

# BiGRU + Attention with UNI
bigru_att_model, history_bigru_att, best_bigru_att_config, bigru_att_results_df = run_rnn_sweep(
    model_tag="bigru_att_uni", 
    model_builder=build_bigru_attention_classifier,
    rnn_sweep=bigru_att_sweep,
    train_ds=uni_rnn_train,
    val_ds=uni_rnn_val,
    y=y,
    vocab_size=uni_vocab_size
)

# BiLSTM with UNI
bilstm_model, history_bilstm, best_bilstm_config, bilstm_results_df = run_rnn_sweep(
    model_tag="bilstm_uni",
    model_builder=build_bilstm_classifier, 
    rnn_sweep=bilstm_sweep, 
    train_ds=uni_rnn_train, 
    val_ds=uni_rnn_val, 
    y=y, 
    vocab_size=uni_vocab_size
)

# OPPOSITE TOKENIZER RUNS

# Transformer with UNI
att_uni_model, history_att_uni, best_att_uni_config, att_uni_results_df = run_transformer_sweep(
    model_tag="att_uni", 
    sweep=sweep, 
    y=y, 
    train_data=uni_train_data, 
    val_data=uni_val_data, 
    vocab_size=uni_vocab_size
)

# Plain BiGRU with BPE
bigru_bpe_model, history_bigru_bpe, best_bigru_bpe_config, bigru_bpe_results_df = run_rnn_sweep(
    model_tag="bigru_bpe", 
    model_builder=build_bigru_classifier, 
    rnn_sweep=rnn_sweep, 
    train_ds=bpe_rnn_train, 
    val_ds=bpe_rnn_val, 
    y=y, 
    vocab_size=bpe_vocab_size
)

# BiGRU + Attention with BPE
bigru_att_bpe_model, history_bigru_att_bpe, best_bigru_att_bpe_config, bigru_att_bpe_results_df = run_rnn_sweep(
    model_tag="bigru_att_bpe", 
    model_builder=build_bigru_attention_classifier, 
    rnn_sweep=bigru_att_sweep, 
    train_ds=bpe_rnn_train, 
    val_ds=bpe_rnn_val,
    y=y,
    vocab_size=bpe_vocab_size
)

# BiLSTM with BPE
bilstm_bpe_model, history_bilstm_bpe, best_bilstm_bpe_config, bilstm_bpe_results_df = run_rnn_sweep(
    model_tag="bilstm_bpe", 
    model_builder=build_bilstm_classifier, 
    rnn_sweep=bilstm_sweep, 
    train_ds=bpe_rnn_train, 
    val_ds=bpe_rnn_val, 
    y=y, 
    vocab_size=bpe_vocab_size
)

In [ ]:
class_names = [emotion_map[index] for index in sorted(emotion_map.keys())]

custom_objects = {
    "TokenAndPositionEmbedding": TokenAndPositionEmbedding, "TransformerBlock": TransformerBlock, "TokenMask": TokenMask, "StripMask": StripMask, "MaskedAttentionPooling": MaskedAttentionPooling
}

model_runs = [
    {"display_name": "Transformer + BPE", "results_csv": "att_bpe_v2_results.csv", "test_data": bpe_test_data}, 
    {"display_name": "BiGRU + UNI", "results_csv": "bigru_uni_v2_results.csv", "test_data": uni_rnn_test}, 
    {"display_name": "BiGRU + Attention + UNI", "results_csv": "bigru_att_uni_v2_results.csv", "test_data": uni_rnn_test},
    {"display_name": "BiLSTM + UNI", "results_csv": "bilstm_uni_v2_results.csv", "test_data": uni_rnn_test},
    {"display_name": "Transformer + UNI", "results_csv": "att_uni_v2_results.csv", "test_data": uni_test_data}, 
    {"display_name": "BiGRU + BPE", "results_csv": "bigru_bpe_v2_results.csv", "test_data": bpe_rnn_test}, 
    {"display_name": "BiGRU + Attention + BPE", "results_csv": "bigru_att_bpe_v2_results.csv", "test_data": bpe_rnn_test}, 
    {"display_name": "BiLSTM + BPE", "results_csv": "bilstm_bpe_v2_results.csv", "test_data": bpe_rnn_test}
]

def get_y_true_from_dataset(dataset):
    y_true = []

    for batch in dataset:
        X_batch, y_batch = batch
        y_true.extend(y_batch.numpy())

    return np.array(y_true)

summary_rows = []

for model_info in model_runs:
    display_name = model_info["display_name"]
    results_path = os.path.join(log_dir, model_info["results_csv"])
    test_data = model_info["test_data"]

    print("\n" + "=" * 80)
    print(display_name)
    print("=" * 80)

    if not os.path.exists(results_path):
        print(f"Missing results file: {results_path}")
        continue

    results_df = pd.read_csv(results_path).sort_values("macro_f1", ascending=False)

    if results_df.empty:
        print("Results file exists but is empty.")
        continue

    best_row = results_df.iloc[0]
    best_run_name = best_row["run_name"]
    best_macro_f1 = best_row["macro_f1"]
    best_model_path = os.path.join(log_dir, best_run_name, "best.keras")

    print(f"Best run: {best_run_name}")
    print(f"Validation macro F1 from CSV: {best_macro_f1:.4f}")
    print(f"Model path: {best_model_path}")

    if not os.path.exists(best_model_path):
        print("Best model file not found.")
        continue

    model = tf.keras.models.load_model(    best_model_path, compile=False, custom_objects=custom_objects)

    y_true = get_y_true_from_dataset(test_data)
    y_pred_probs = model.predict(test_data, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    print("\nClassification report")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    conf_matrix = confusion_matrix(y_true, y_pred, labels=list(sorted(emotion_map.keys())))

    conf_matrix_df = pd.DataFrame(conf_matrix, index=[f"true_{name}" for name in class_names], columns=[f"pred_{name}" for name in class_names])

    print("Confusion matrix with labels")
    print(conf_matrix_df)

    plt.figure(figsize=(8, 6))
    display = ConfusionMatrixDisplay(    confusion_matrix=conf_matrix, display_labels=class_names)
    display.plot(    cmap="Blues", xticks_rotation=45, values_format="d")
    plt.title(f"{display_name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

    summary_rows.append({
        "model": display_name, "run_name": best_run_name, "val_macro_f1_from_csv": best_macro_f1
    })

summary_df = pd.DataFrame(summary_rows).sort_values("val_macro_f1_from_csv", ascending=False)
print("\nOverall summary")
print(summary_df)

In [ ]:
# TOODs
# Stop words such as "the," "a," "an," "in," "and"
# Data Augmentation 